In [1]:
import requests
# import pandas as pd

from datetime import datetime, timedelta


In [2]:

import sys
import os

root = os.path.abspath(os.path.join(os.getcwd(), "../"))

if root not in sys.path:
    sys.path.insert(0, root)

print(root)

/home/hilaneto/Jhan/Trabalho/python/projetos/AtualizaIndicador


#### Busca Dados IGP-M via código Atualiza-Indicador

In [ ]:

from database.conexao import conectar
from indicadores.igpm import Igpm

with conectar():
    dados_igpm = Igpm.buscar()
    for aa in dados_igpm:
        print(aa)


#### Busca Dados IGP-M via código Atualiza-Indicador

In [ ]:

from database.conexao import conectar
from indicadores.igpm import Igpm

with conectar():
    dados_igpm = Igpm.select()
    for aa in dados_igpm:
        print(aa.indice)
        print(aa.dt_referencia)


#### Busca Dados IGP-M Diretamente da API

In [19]:

ultimos_meses = 5

# API Banco Central -------------------------------------------
url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.189/dados/ultimos/{ultimos_meses}?formato=json"

resposta = requests.get(url, timeout=10)
resposta.raise_for_status()
dados = resposta.json() # lista de dicionários

# Percorrer a lista de dicionário
for igpm in dados:
    print(igpm)


{'data': '01/04/2026', 'valor': '2.73'}
{'data': '01/05/2026', 'valor': '0.84'}
{'data': '01/06/2026', 'valor': '-0.50'}
{'data': '01/07/2026', 'valor': '-1.16'}
{'data': '01/08/2026', 'valor': '-0.22'}


#### Gera datas

In [ ]:

# Datas -------------------------------------------------------
hoje = datetime.now()

data_inicial = (hoje - timedelta(days=230)).strftime("%d/%m/%Y")
data_final = hoje.strftime("%d/%m/%Y")

print(f"Período: {data_inicial} até {data_final}")


#### Insere Dados no Banco (transformando dados)

In [ ]:

# API Banco Central -------------------------------------------
url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.189/dados?formato=json&dataInicial={data_inicial}&dataFinal={data_final}"

resposta = requests.get(url, timeout=10)
resposta.raise_for_status() # Valida a resposta HTTP; se houver erro, não deixa o código continuar.
dados = resposta.json()     # lista de dicionários


# Transforma dados para o formato da tabela Igpm
dados_igpm = []
for registro in dados:
    dados_igpm.append({"indice": registro["valor"],
                       "status": True,
                       "dt_referencia": datetime.strptime(registro["data"], "%d/%m/%Y").date()
                      })

with conectar():
    Igpm.insert_many(dados_igpm).on_conflict_ignore().execute()

#### Visualiza DADOS IGP-M (lista de dicionário)

In [ ]:
for igpm in dados:
    print(igpm["data"], igpm["valor"])


#### Carga direta no Banco (registro por registro)

In [ ]:

from database.conexao import conectar
from indicadores.igpm import Igpm
from decimal import Decimal

with conectar():
    for registro in dados:
        Igpm.insert(indice=registro["valor"], status=True, dt_referencia=registro["data"]).on_conflict_ignore().execute()


#### Carga direta no Banco (Em Massa)

In [ ]:

from database.conexao import conectar
from indicadores.igpm import Igpm

with conectar():
    Igpm.insert_many(dados_igpm).on_conflict_ignore().execute()

#### Atualiza tabela IGP-M usando método do código Atualiza-Indicador

In [ ]:

from database.conexao import conectar
from indicadores.igpm import Igpm

aa = Igpm.atualizar_igpm()

print(aa)